# Modélisation Prédictive des Prix Immobiliers (IDF)
**Objectif :** Développer un algorithme capable d'estimer la valeur foncière d'un bien en Île-de-France à partir des données historiques DVF.

---
## 1. Initialisation de l'Environnement
Nous importons les bibliothèques nécessaires à la manipulation de données (`pandas`), au calcul numérique (`numpy`), à la visualisation (`matplotlib`, `seaborn`) et au Machine Learning (`scikit-learn`, `xgboost`).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Configuration
load_dotenv(dotenv_path='../.env')
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

✅ Environnement configuré.


## 2. Extraction des Données (ETL)
Nous récupérons les données depuis notre base PostgreSQL. 
**Stratégie de filtrage :**
* Focus sur le marché **résidentiel** (Maisons et Appartements).
* Exclusion des transactions extrêmes (< 10 000 € et > 2 000 000 €) pour éviter que les "outliers" (châteaux ou ventes symboliques) ne polluent l'apprentissage du modèle.

In [10]:
query = """
SELECT date_mutation, valeur_fonciere, code_postal, surface_reelle_bati, 
       nombre_pieces_principales AS nb_pieces, surface_terrain, type_local
FROM transactions
WHERE surface_reelle_bati > 0 
  AND valeur_fonciere BETWEEN 10000 AND 2000000 
  AND type_local IN ('Appartement', 'Maison')
"""
df = pd.read_sql(text(query), engine)
print(f"{len(df):,} transactions récupérées.")

769,510 transactions récupérées.


## 3. Feature Engineering (Ingénierie des Variables)
Nous transformons les données brutes en informations exploitables par l'IA :

1. **Dimension Temporelle :** Extraction de l'année et du mois pour capter l'inflation et les cycles saisonniers du marché immobilier.
2. **Indice de Standing :** Calcul du ratio `surface_par_piece` pour différencier les grands volumes luxueux des petits espaces denses.
3. **Optimisation Géographique (Target Encoding) :** Au lieu de créer des centaines de colonnes pour chaque code postal (ce qui saturerait la mémoire), nous remplaçons chaque code postal par le **prix moyen historique** du quartier.

In [ ]:
# Temps
df['date_mutation'] = pd.to_datetime(df['date_mutation'])
df['annee'] = df['date_mutation'].dt.year
df['mois'] = df['date_mutation'].dt.month

# Standing (Ratio surface/pièce)
df['surface_par_piece'] = df['surface_reelle_bati'] / df['nb_pieces'].replace(0, 1)

# Target Encoding (Géographie)
postal_means = df.groupby('code_postal')['valeur_fonciere'].mean()
df['postal_score'] = df['code_postal'].map(postal_means)

# Nettoyage final
df['surface_terrain'] = df['surface_terrain'].fillna(0)
df_ml = pd.get_dummies(df.drop(['date_mutation', 'code_postal'], axis=1), columns=['type_local'], drop_first=True)

Feature Engineering terminé.


## 4. Entraînement du Modèle : "Voting Regressor"
Après avoir testé plusieurs algorithmes (Random Forest, XGBoost), j'ai opté pour une approche par **Ensemble Learning**. 

Nous utilisons un `VotingRegressor` qui combine :
* **Random Forest :** Robuste et stable, il capte bien les tendances générales.
* **XGBoost :** Algorithme de "Boosting" ultra-performant qui se focalise sur la correction des erreurs résiduelles.

L'union de ces deux modèles permet d'obtenir une prédiction plus équilibrée.

In [ ]:
X = df_ml.drop('valeur_fonciere', axis=1)
y = df_ml['valeur_fonciere']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Définition des modèles
rf = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1)
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8, n_jobs=-1)

# Vote majoritaire
model = VotingRegressor([('rf', rf), ('xgb', xgb)])
model.fit(X_train, y_train)

Modèle entraîné.


## 5. Évaluation des Performances
Nous utilisons deux métriques clés pour juger la qualité de notre IA :
* **R² (Coefficient de détermination) :** Représente le pourcentage de la réalité expliqué par le modèle.
* **MAE (Mean Absolute Error) :** Représente l'erreur moyenne en euros sur chaque transaction.

> **Note :** On constate un plafond de verre autour de 0.75. En effet, des facteurs cruciaux comme l'étage, l'exposition, la vue ou l'état intérieur (travaux) ne sont pas présents dans les données publiques.

In [14]:
preds = model.predict(X_test)
print(f"Score R² Final : {r2_score(y_test, preds):.4f}")
print(f"Erreur Moyenne (MAE) : {mean_absolute_error(y_test, preds):,.0f} €")

Score R² Final : 0.7484
Erreur Moyenne (MAE) : 86,646 €


## 6. Sérialisation et Export
Pour rendre notre modèle utilisable dans une application réelle (API ou Interface Web), nous sauvegardons :
1. Le **modèle entraîné**.
2. Le **dictionnaire de prix moyens** par code postal (nécessaire pour transformer les futures entrées utilisateurs).
3. La **liste des colonnes** pour garantir la cohérence des données entrantes.

In [ ]:
# Création du dossier d'export s'il n'existe pas
os.makedirs('../models', exist_ok=True)

# Sauvegarde du modèle 
joblib.dump(model, '../models/rf_model_immo.joblib')

# Sauvegarde de la mémoire géographique (Target Encoding)
joblib.dump(postal_means, '../models/postal_means.joblib')

# Sauvegarde de la structure des colonnes
expected_columns = list(X_train.columns)
joblib.dump(expected_columns, '../models/expected_columns.joblib')

Sauvegarde du modèle prédictif...
Sauvegarde du dictionnaire des codes postaux...
Sauvegarde de la structure des variables...

EXPORT TERMINÉ ! Les fichiers .joblib sont prêts dans le dossier '/models'.
